In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print('using', DEVICE)

using mps


In [2]:
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [3]:
next(iter(train_dataloader))[1]

tensor([6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6,
        2, 6, 3, 5, 4, 0, 0, 9, 1, 3, 4, 0, 3, 7, 3, 3, 5, 2, 2, 7, 1, 1, 1, 2,
        2, 0, 9, 5, 7, 9, 2, 2, 5, 2, 4, 3, 1, 1, 8, 2])

In [4]:
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding = 'same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Flatten(),
            nn.Linear(32*32*32//4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )
    def forward(self, input):
        logits = self.model(input)
        return logits

In [5]:
model = CustomCNN().to(DEVICE)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)
optimizer = torch.optim.RMSprop(model.parameters(), lr=LR)

In [ ]:
for epoch in range(1, EPOCHS+1):
    train_loss = 0
    model.train()
    for batch_size, (X, y) in enumerate(train_dataloader):
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    test_loss = 0
    model.eval()
    for batch_size, (X, y) in enumerate(test_dataloader):
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        loss = loss_fn(pred, y)
        test_loss += loss.item()
    print(epoch, train_loss / len(training_data), test_loss / len(test_data))

Epoch: 1 Train: 0.0285635462808609 Test: 0.024899517679214476
Epoch: 2 Train: 0.022014634815454483 Test: 0.022033023285865785
Epoch: 3 Train: 0.02008520494222641 Test: 0.019050463378429413
Epoch: 4 Train: 0.018867352851629257 Test: 0.017883144915103913
Epoch: 5 Train: 0.017834648555517196 Test: 0.018382415533065796
Epoch: 6 Train: 0.01698879983305931 Test: 0.01694105414748192
Epoch: 7 Train: 0.016170969918966292 Test: 0.017665112233161927
Epoch: 8 Train: 0.015551714347600937 Test: 0.01692960832118988
Epoch: 9 Train: 0.015039867683649063 Test: 0.015698939418792725
Epoch: 10 Train: 0.014446474907398223 Test: 0.01750242359638214
Epoch: 11 Train: 0.013940135718584061 Test: 0.017189049208164216
Epoch: 12 Train: 0.013502608159780502 Test: 0.01592475209236145
Epoch: 13 Train: 0.013171149096488952 Test: 0.015853076994419096
Epoch: 14 Train: 0.012784588900804519 Test: 0.01679360842704773
Epoch: 15 Train: 0.012468881692886352 Test: 0.016131028175354004
Epoch: 16 Train: 0.01209687178492546 Test: 